In [ ]:
# Colab setup: install pyedautils; elsewhere load plotly.js from a CDN for the plots
import sys
if "google.colab" in sys.modules:
    %pip install -q "pyedautils>=0.0.32"
else:
    import plotly.io as pio
    pio.renderers.default = "notebook_connected"

# Calendar Heat Map

```{admonition} Goal
:class: tip
Visualize daily electricity consumption as a calendar heatmap to identify patterns and anomalies over time.
```

```{admonition} Data Basis
:class: note
The dataset `flatElectricity.csv` contains cumulative electricity meter readings at 15-minute intervals for multiple flats. We use the column `FlatA_Ele`, convert cumulative readings to consumption by computing differences and sum them per day. Days with missing readings stay empty in the calendar.
```

## Data Preparation

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/flatElectricity.csv"
raw = pd.read_csv(url, sep=";")

# Parse timestamp and set as index
raw["time"] = pd.to_datetime(raw["time"])
raw = raw.set_index("time")

# Consumption per 15 minutes from the cumulative meter readings (kWh)
consumption = raw["FlatA_Ele"].diff()

# Daily sums. min_count=96: a day counts only if all 96 quarter-hours have a reading;
# otherwise sum() would turn days with missing data into (too) low or zero consumption
daily = consumption.resample("D").sum(min_count=96).dropna()

# Build clean 2-column DataFrame [timestamp, value]
df = daily.reset_index()
df.columns = ["timestamp", "value"]
df.head()

## Visualization

In [ ]:
from pyedautils.plots.heatmaps import plot_heatmap_calendar

fig = plot_heatmap_calendar(df, title="Electricity — Calendar Heat Map", ylab="Electricity [kWh/d]")
fig.show()

::::{dropdown} Customization
The `plot_heatmap_calendar` function accepts a `colorscale` parameter to change the color palette of the heatmap. For example:

```python
fig = plot_heatmap_calendar(df, title="Electricity — Calendar Heat Map", ylab="Electricity [kWh/d]", colorscale="Viridis")
```
::::